# Hybrid Bioprocess ML: Reproducible Synthetic Study

This notebook demonstrates a constrained hybrid model for fed-batch mammalian cell culture. It is a portfolio-quality synthetic study: results are useful for evaluating methodology, but they are **not** evidence from an industrial process.

## Study Contract

The learned component is a bounded multiplier on mechanistic growth. Every candidate is evaluated on held-out batches and scientific constraints are gates, not penalty terms. The later sections quantify variation across predeclared synthetic seeds and bootstrap-resampled training batches.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().resolve().parent
if not (project_root / "src").exists():
    project_root = Path.cwd().resolve()
sys.path.insert(0, str(project_root / "src"))

import hybridbio

importlib.reload(hybridbio)

from hybridbio import (
    EnsembleConfig,
    KineticParameters,
    StudyConfig,
    TrainingConfig,
    generate_dataset,
    run_repeated_study,
    train_and_evaluate,
    train_bootstrap_ensemble,
    train_test_split_batches,
)
from hybridbio.mechanistic import STATE_NAMES

params = KineticParameters()
training_config = TrainingConfig(seed=7)
print(f"Project root: {project_root}")

ImportError: cannot import name 'EnsembleConfig' from 'hybridbio' (/Users/kemal/Desktop/codes/ml/hybrid-bioprocess-lab/src/hybridbio/__init__.py)

## Held-Out Baseline Comparison

Data are split by whole batch, never by timepoint. This prevents trajectory leakage between training and test partitions.

In [ ]:
batches = generate_dataset(n_batches=18, seed=7)
train_batches, test_batches = train_test_split_batches(batches, n_test=4)
hybrid, hybrid_report, baseline_report = train_and_evaluate(
    train_batches,
    test_batches,
    p=params,
    cfg=training_config,
)

print(baseline_report.render())
print()
print(hybrid_report.render())
print()
print(
    "Hybrid minus baseline NRMSE: "
    f"{hybrid_report.metrics['nrmse_mean'] - baseline_report.metrics['nrmse_mean']:+.4f}"
)

============================= test session starts ==============================
platform darwin -- Python 3.14.6, pytest-9.1.1, pluggy-1.6.0
rootdir: /Users/kemal/Desktop/codes/ml/hybrid-bioprocess-lab
configfile: pyproject.toml
testpaths: tests
plugins: anyio-4.14.2
collected 47 items

tests/test_inference.py .....                                            [ 10%]
tests/test_mechanistic.py ...........                                    [ 34%]
tests/test_registry.py ..                                                [ 38%]
tests/test_regression.py .........                                       [ 57%]
tests/test_reporting.py ...                                              [ 63%]
tests/test_rollout.py ..                                                 [ 68%]
tests/test_scientific_constraints.py ..........                          [ 89%]
tests/test_torch_correction.py .....                                     [100%]

=============================== warnings summary =====================

## Trajectory Inspection

A single held-out batch makes the model behavior inspectable. Accuracy is reported alongside constraints; neither replaces the other.

In [ ]:
batch = test_batches[0]
model_for_batch = hybrid.__class__(
    params=hybrid.params,
    feed=batch.feed,
    correction=hybrid.correction,
    t_end_h=hybrid.t_end_h,
    dt_h=hybrid.dt_h,
)
t_pred, y_pred = model_for_batch.simulate(batch.y0)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for axis, state in zip(axes.ravel(), ("Xv", "S", "L", "P"), strict=True):
    state_index = STATE_NAMES.index(state)
    axis.plot(batch.t, batch.Y[:, state_index], "o", label="observed", alpha=0.7)
    axis.plot(t_pred, y_pred[:, state_index], label="hybrid")
    axis.set_title(state)
    axis.set_xlabel("time [h]")
    axis.legend()
fig.suptitle(f"Held-out batch {batch.batch_id}")
fig.tight_layout()

train batches: 18
test batches:  6

=== Hybrid model ===
Evaluation over 6 batch(es)
----------------------------------------------
  final_titre_rel_err            0.0474
  nrmse_L                        0.0289
  nrmse_P                        0.0280
  nrmse_S                        0.1212
  nrmse_Xv                       0.0560
  nrmse_mean                     0.0586
----------------------------------------------
  scientific constraints              0 violation(s)
  verdict                          PASS

=== Mechanistic-only baseline ===
Evaluation over 6 batch(es)
----------------------------------------------
  final_titre_rel_err            0.0595
  nrmse_L                        0.0367
  nrmse_P                        0.0422
  nrmse_S                        0.1612
  nrmse_Xv                       0.0695
  nrmse_mean                     0.0774
----------------------------------------------
  scientific constraints              0 violation(s)
  verdict                          PAS

## Repeated-Seed Evidence

A single favorable split is not enough. The study repeats paired, held-out comparisons over predeclared seeds and computes a bootstrap confidence interval from batch-level deltas. A result counts as an improvement only when every run is admissible and the complete interval is below zero.

In [ ]:
study_result = run_repeated_study(
    StudyConfig(
        seeds=(7, 17, 29),
        n_batches=12,
        n_test=3,
        n_bootstrap=500,
        bootstrap_seed=7,
    ),
    params=params,
    training_config=training_config,
)
ci = study_result.nrmse_delta
print(f"Paired NRMSE delta: {ci.estimate:+.4f}")
print(f"{ci.confidence:.0%} bootstrap CI: [{ci.lower:+.4f}, {ci.upper:+.4f}]")
print(f"Admissible runs: {study_result.admissible_runs}/{study_result.n_runs}")
print(f"Evidence supports improvement: {study_result.candidate_improves}")

## Bootstrap Trajectory Uncertainty

The ensemble resamples whole training batches, preserving within-batch correlation. Its quantiles express variability caused by the available synthetic data, not a safety guarantee for real manufacturing.

In [ ]:
ensemble = train_bootstrap_ensemble(
    train_batches,
    params=params,
    training_config=training_config,
    config=EnsembleConfig(n_members=6, seed=7),
)
interval = ensemble.predict(batch.y0, batch.feed)
titre_index = STATE_NAMES.index("P")
coverage = interval.empirical_coverage(batch.Y)

fig, axis = plt.subplots(figsize=(10, 4))
axis.plot(batch.t, batch.Y[:, titre_index], "o", label="observed titre")
axis.plot(interval.t, interval.median[:, titre_index], label="ensemble median")
axis.fill_between(
    interval.t,
    interval.lower[:, titre_index],
    interval.upper[:, titre_index],
    alpha=0.25,
    label="90% bootstrap interval",
)
axis.set(xlabel="time [h]", ylabel="product titre [mg/L]")
axis.set_title(f"Held-out empirical coverage across all states: {coverage:.1%}")
axis.legend()
fig.tight_layout()

## Data and Workflow Boundaries

External CSV exports enter through `hybridbio.ingestion.load_batches_csv()`, which validates explicit units, finite values, monotonic timestamps, non-negative state values, and stable per-batch feed settings. The Optuna and Ray workflows prune or exclude scientifically inadmissible trials rather than treating a violation as a worse numeric score.

Before process decisions, replace this synthetic plant with versioned real data, prespecify stress scenarios and acceptance criteria, calibrate intervals on external batches, and retain audit artifacts for every promoted model.